# Relatorios Gold

Este notebook le a Silver em batch e prepara os indicadores analiticos da camada Gold.

- **Atividade:** quantidade de aeronaves distintas observadas por data e hora.
- **Altitude e velocidade:** medias por data e hora, sem confundir snapshots com voos.
- **Companhia aerea:** somente sera calculado se houver uma dimensao explicita validada.
- **Duracao na area:** depende de eventos de entrada/saida ou de uma regra de sessao aprovada.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf

spark = (SparkSession.builder
         .appName("GoldReportsNotebook")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
SILVER_PATH = str(BASE_DIR / "data" / "silver" / "output")
RUN_DATE = None  # Exemplo: "2026-08-19"
TIMEZONE = "UTC"

spark.conf.set("spark.sql.session.timeZone", TIMEZONE)
print(f"Silver: {SILVER_PATH}")
print(f"Timezone: {TIMEZONE}")
print(f"Filtro de data: {RUN_DATE or 'todos os dados'}")

Silver: /home/jovyan/work/data/silver/output
Timezone: UTC
Filtro de data: todos os dados


In [4]:
required_columns = {
    "time", "icao24", "baro_altitude", "velocity",
    "on_ground", "year", "month", "day", "Nome", "ICAO"
}

try:
    silver = spark.read.parquet(SILVER_PATH)
except Exception as error:
    raise RuntimeError(
        f"Nao foi possivel ler a Silver em {SILVER_PATH}. "
        "Verifique se o streaming Silver terminou de gravar Parquet valido."
    ) from error

missing_columns = required_columns.difference(silver.columns)
if missing_columns:
    raise ValueError(f"Contrato Silver incompleto; faltam: {sorted(missing_columns)}")

time_type = silver.schema["time"].dataType.simpleString()
if time_type != "timestamp":
    raise TypeError(
        f"A coluna time deve ser timestamp na Silver; tipo encontrado: {time_type}"
    )

if RUN_DATE:
    silver = silver.filter(
        sf.to_date("time") == sf.to_date(sf.lit(RUN_DATE))
    )

silver = silver.filter(
    sf.col("icao24").isNotNull() & sf.col("time").isNotNull()
)

print("Schema validado:")
silver.printSchema()
print(f"Registros no recorte: {silver.count()}")

Schema validado:
root
 |-- time: timestamp (nullable = true)
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- origin_country: string (nullable = true)
 |-- time_position: long (nullable = true)
 |-- last_contact: long (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- baro_altitude: double (nullable = true)
 |-- on_ground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- true_track: double (nullable = true)
 |-- vertical_rate: double (nullable = true)
 |-- geo_altitude: double (nullable = true)
 |-- spi: boolean (nullable = true)
 |-- position_source: integer (nullable = true)
 |-- category: integer (nullable = true)
 |-- Airline ID: string (nullable = true)
 |-- Nome: string (nullable = true)
 |-- Alias: string (nullable = true)
 |-- IATA: string (nullable = true)
 |-- ICAO: string (nullable = true)
 |-- Indicativo: string (nullable = true)
 |-- País: string (nullable = true)
 

Registros no recorte: 1069


## 1. Aeronaves ativas por horario

Cada linha representa a quantidade de `icao24` distintos com ao menos um snapshot valido naquele horario. Isso mede atividade observada, nao quantidade de voos.

In [ ]:
activity = (
    silver
    .filter(sf.col("icao24").isNotNull())
    .withColumn("date", sf.to_date("time"))
    .withColumn("hour", sf.hour("time"))
    .groupBy("date", "hour")
    .agg(sf.countDistinct("icao24").alias("active_aircraft_count"))
    .withColumn("year", sf.year("date"))
    .withColumn("month", sf.month("date"))
    .withColumn("day", sf.dayofmonth("date"))
    .orderBy("date", "hour")
)

display(activity)
activity.show()

## 2. Altitude e velocidade medias

As unidades seguem a origem OpenSky: altitude em metros e velocidade em metros por segundo. Valores nulos sao ignorados pelo `avg`; registros em solo permanecem incluidos nesta primeira leitura.

In [ ]:
altitude_speed = (
    silver
    .withColumn("date", sf.to_date("time"))
    .withColumn("hour", sf.hour("time"))
    .groupBy("date", "hour")
    .agg(
        sf.round(sf.avg("baro_altitude"),2).alias("avg_altitude"),
        sf.avg("velocity").alias("avg_velocity"),
        sf.count("baro_altitude").alias("altitude_observations"),
        sf.count("velocity").alias("velocity_observations"),
    )
    .orderBy("date", "hour")
)

display(altitude_speed)
altitude_speed.show()

## 3. Companhia aerea

A Silver possui a dimensao de companhias enriquecida pelas colunas `Nome` e `ICAO`. A metrica conta `icao24` distintos observados por companhia e data; representa aeronaves monitoradas, nao quantidade de voos.

## 4. Duracao media na area monitorada

Uma nova sessao comeca quando a lacuna entre aparicoes do mesmo `icao24` supera 30 minutos. A media usa somente sessoes fechadas com duracao positiva.

In [ ]:
airline_counts = (
    silver
    .withColumn("date", sf.to_date("time"))
    .withColumn("airline_name", sf.coalesce(sf.nullif(sf.trim(sf.col("Nome")), sf.lit("")), sf.lit("Desconhecida")))
    .withColumn("airline_icao", sf.coalesce(sf.nullif(sf.trim(sf.col("ICAO")), sf.lit("")), sf.lit("N/A")))
    .groupBy("date", "airline_icao", "airline_name")
    .agg(sf.countDistinct("icao24").alias("aircraft_count"))
    .orderBy("date", "airline_name")
)

display(airline_counts)
airline_counts.show()

In [ ]:
from pyspark.sql.window import Window

SESSION_GAP_SECONDS = 30 * 60

observation_window = Window.partitionBy("icao24").orderBy("time")
session_window = (
    Window.partitionBy("icao24")
    .orderBy("time")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

session_observations = (
    silver
    .withColumn("previous_time", sf.lag("time").over(observation_window))
    .withColumn("next_time", sf.lead("time").over(observation_window))
    .withColumn(
        "new_session",
        sf.when(
            sf.col("previous_time").isNull()
            | (
                sf.col("time").cast("long")
                - sf.col("previous_time").cast("long")
                > SESSION_GAP_SECONDS
            ),
            1,
        ).otherwise(0),
    )
    .withColumn("session_id", sf.sum("new_session").over(session_window))
    .withColumn(
        "session_closes",
        sf.when(
            sf.col("next_time").isNotNull()
            & (
                sf.col("next_time").cast("long")
                - sf.col("time").cast("long")
                > SESSION_GAP_SECONDS
            ),
            1,
        ).otherwise(0),
    )
)

sessions = (
    session_observations
    .groupBy("icao24", "session_id")
    .agg(
        sf.min("time").alias("entry_ts"),
        sf.max("time").alias("exit_ts"),
        sf.max("session_closes").alias("has_exit"),
        sf.first("Nome", ignorenulls=True).alias("airline_name"),
        sf.first("ICAO", ignorenulls=True).alias("airline_icao"),
    )
    .withColumn(
        "duration_seconds",
        sf.col("exit_ts").cast("long") - sf.col("entry_ts").cast("long"),
    )
    .withColumn("entry_date", sf.to_date("entry_ts"))
)

closed_sessions = sessions.filter(
    (sf.col("has_exit") == 1) & (sf.col("duration_seconds") > 0)
)

duration_report = (
    closed_sessions
    .groupBy("entry_date")
    .agg(
        sf.round(sf.avg("duration_seconds"), 2).alias("avg_area_duration_seconds"),
        sf.count("*").alias("sessions_used"),
    )
    .orderBy("entry_date")
)

session_quality = sessions.agg(
    sf.count("*").alias("sessions_total"),
    sf.sum(sf.when(sf.col("has_exit") == 1, 1).otherwise(0)).alias("sessions_closed"),
    sf.sum(sf.when(sf.col("has_exit") == 0, 1).otherwise(0)).alias("sessions_open"),
    sf.sum(
        sf.when(
            (sf.col("has_exit") == 1) & (sf.col("duration_seconds") <= 0), 1
        ).otherwise(0)
    ).alias("sessions_discarded_non_positive"),
)

print(f"Regra aplicada: nova sessao quando gap > {SESSION_GAP_SECONDS} segundos")
display(duration_report)
duration_report.show()
display(session_quality)
session_quality.show()

### Detalhamento das sessoes fechadas

A tabela abaixo permite confirmar individualmente as entradas, saidas e duracoes usadas no calculo da media.

In [11]:
display(
    closed_sessions.select(
        "icao24", "airline_icao", "airline_name",
        "entry_ts", "exit_ts", "duration_seconds"
    ).orderBy("entry_ts")
)

NameError: name 'closed_sessions' is not defined